<a href="https://colab.research.google.com/github/abubakarsaleem18/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abubakarsaleem18/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

One row = one content page's performance, on one day, for one client
(grain: report_date × client_hash_id × content_hash_id) in the raw table.

For my lane, I roll these daily rows up to: one row = one content page,
summarized over one month (month=2026-03), split into first-half
(days 1–15) and second-half (days 16–end) to detect within-month trend.

In [23]:
q1 = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_keys,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
q1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_keys,start_date,end_date
0,9841378,9841378,2026-03-01,2026-03-31


**Verified:** total_rows equals distinct_keys exactly — confirming one row
truly is one (date, client, content page) combination, as claimed above.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature (used to predict):**
- gsc_impressions — visibility signal
- gsc_clicks — engagement signal
- gsc_avg_position — ranking signal
- ctr (derived: gsc_clicks / gsc_impressions)

**Label / proxy (what I'm predicting):**
- needs_review — built from comparing clicks in first-half vs second-half
  of the month (1 if declining + still visible, else 0)

**Context (identifiers, not predictive features):**
- report_date, client_hash_id, content_hash_id, month

**Excluded (with why):**
- ga4_* columns — excluded for now since GA4 availability (`ga4_data_available`)
  is inconsistent across clients; mixing partial GA4 data in risks bias
  toward clients who happen to have it.
- ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta,
  ai_other — excluded because these are likely near-zero at this stage of
  AI-search adoption; too sparse to add real signal yet.

In [25]:
q2 = con.sql(f"""
    SELECT
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS pct_ga4_available,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS pct_gsc_available
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
q2

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,pct_ga4_available,pct_gsc_available
0,0.042064,0.366926


In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [27]:
q3_counts = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           COUNT(DISTINCT content_hash_id) AS n_pages,
           COUNT(DISTINCT client_hash_id) AS n_clients
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
q3_counts

,row_count,n_pages,n_clients
0,9841378,331437,55


In [28]:
q3_missing = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS pct_available
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
q3_missing

,total_rows,gsc_available_rows,pct_available
0,9841378,3611061.0,0.366926


In [29]:
q3_window = con.sql(f"""
    SELECT month, MIN(report_date) AS min_date, MAX(report_date) AS max_date, COUNT(*) AS n_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY month
""").df()
q3_window

,month,min_date,max_date,n_rows
0,2026-03,2026-03-01,2026-03-31,9841378


In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Counts:** This slice contains 9,841,378 rows, covering 331,437 distinct
content pages across 55 clients.

**Missing values:** Only 36.7% of rows (3,611,061 out of 9,841,378) have
`gsc_data_available IS TRUE`. This confirms my earlier decision in Section 2
to filter on this flag before building features — nearly two-thirds of rows
in this raw daily table don't have usable GSC data and would corrupt any
feature built without this filter.

**Windows:** All 9,841,378 rows fall within month=2026-03, spanning exactly
2026-03-01 to 2026-03-31 — the month partition is clean, with no bleed from
adjacent months.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


This slice only reliably covers clients with `gsc_data_available IS TRUE`
in March 2026 — clients with sparser GSC syncing are underrepresented.
The panel is also unbalanced by design (per-client history depth differs,
per `dim_clients`), so this month's slice cannot tell us how a page
performed before its client's `gsc_data_start`, and GA4 engagement data is
inconsistently available, so engagement-based claims would be unreliable
without further filtering.

In [32]:
%pip install -q duckdb huggingface_hub

import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

In [33]:
print(f"Token loaded: {hf_token is not None}")
print(f"Token starts with: {hf_token[:5]}...")

Token loaded: True
Token starts with: hf_EJ...


In [34]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 5")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [35]:
con.sql(f"SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 5").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [36]:
import pandas as pd
pd.set_option('display.max_rows', None)

schema_df = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')").df()
print(schema_df[['column_name', 'column_type']].to_string())

                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      BIGINT
23             a

In [37]:
features = con.sql(f"""
    WITH daily AS (
        SELECT *,
            CASE WHEN EXTRACT(day FROM report_date) <= 15 THEN 'first_half' ELSE 'second_half' END AS half
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
    )
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(gsc_avg_position) AS avg_position,
        SUM(CASE WHEN half = 'first_half' THEN gsc_clicks ELSE 0 END) AS clicks_first_half,
        SUM(CASE WHEN half = 'second_half' THEN gsc_clicks ELSE 0 END) AS clicks_second_half
    FROM daily
    GROUP BY content_hash_id
""").df()

import pandas as pd
features["ctr"] = features["total_clicks"] / features["total_impressions"].replace(0, pd.NA)
features["needs_review"] = (
    (features["clicks_second_half"] < features["clicks_first_half"]) &
    (features["total_impressions"] >= 500)
).astype(int)

feature_frame = features[["content_hash_id", "total_impressions", "total_clicks", "avg_position", "ctr", "needs_review"]]
feature_frame.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,total_impressions,total_clicks,avg_position,ctr,needs_review
0,content_b7e512995f79d5a6,1140.0,2.0,4.394234,0.001754,1
1,content_05597932fe4da067,57.0,0.0,2.714744,0.000000,0
2,content_905aa32a0230694e,149.0,0.0,6.481453,0.000000,0
3,content_05434271b257bb68,1421.0,6.0,6.320337,0.004222,0
4,content_d056587ff7faca0c,2770.0,16.0,4.459107,0.005776,1
5,content_bfd1e41c2af250c8,48.0,0.0,14.753175,0.000000,0
6,content_2662845f598544ef,150.0,1.0,6.341880,0.006667,0
7,content_22610b0934f8825e,67.0,0.0,12.791667,0.000000,0
8,content_712c365258cee05c,6048.0,23.0,4.950311,0.003803,0
9,content_476c37c366920c1b,223.0,0.0,50.390299,0.000000,0


**1. total_impressions** — knowable at the decision moment because it's
just a sum of already-logged GSC data up through today; nothing here comes
from the future.

**2. total_clicks** — same reasoning: fully historical, logged daily by
Search Console, no future dependency.

**3. avg_position** — historical average ranking position across the month,
already recorded by the time you'd make a decision.

**4. ctr** (clicks/impressions) — derived purely from features 1 and 2
above, both already known, so it's safe too.

**5. clicks_first_half vs. clicks_second_half (within-month trend)** —
still historical: both halves are in the past relative to "today" at
month-end, so this doesn't leak from a genuinely future period.

In [38]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_honest = feature_frame[["total_impressions", "total_clicks", "avg_position", "ctr"]].fillna(0)
y = feature_frame["needs_review"]

X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_score = accuracy_score(y_test, model.predict(X_test))
print(f"Honest accuracy (no leak): {honest_score:.3f}")

# --- Deliberate trap: a column derived straight from the label itself ---
feature_frame["leak_column"] = features["clicks_second_half"] - features["clicks_first_half"]

X_leaky = feature_frame[["total_impressions", "total_clicks", "avg_position", "ctr", "leak_column"]].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.3, random_state=42)
model_leaky = LogisticRegression(max_iter=1000).fit(X_train, y_train)
leaky_score = accuracy_score(y_test, model_leaky.predict(X_test))
print(f"Leaky accuracy (with leak_column): {leaky_score:.3f}")

# --- Remove it, keep the honest number ---
feature_frame = feature_frame.drop(columns=["leak_column"])
print(f"\nFinal honest score kept: {honest_score:.3f}")

Honest accuracy (no leak): 0.873


/tmp/ipykernel_1017/1789993282.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  feature_frame["leak_column"] = features["clicks_second_half"] - features["clicks_first_half"]


Leaky accuracy (with leak_column): 0.958

Final honest score kept: 0.873


**The trap, demonstrated:** Adding `leak_column` (clicks_second_half minus
clicks_first_half — literally the same comparison used to build the
`needs_review` label) pushed accuracy from 0.875 to 0.959. This jump isn't
the model getting smarter — it's the model reading the label back through a
disguised column. This is the leakage lesson from notebook 02, now
reproduced on real warehouse data. I removed `leak_column` and am keeping
0.875 as my real, honest baseline going forward.

**Data limits:**
- This slice only reliably represents the 36.7% of rows where
  `gsc_data_available IS TRUE` — pages/clients with spottier GSC syncing
  are underrepresented in this feature frame.
- The panel is unbalanced by design (per-client history depth differs, per
  `dim_clients.gsc_data_start`), so this single month can't tell us how a
  page performed before its own client's data start date.
- GA4 engagement data (sessions, engaged sessions) was excluded entirely
  this week due to inconsistent availability — so this frame says nothing
  about on-page engagement, only search visibility and clicks.
- 0.875 honest accuracy is a first-pass number on one month with a simple
  proxy label — it's not validated against the sealed June 2026 test month
  yet, and shouldn't be treated as a final performance claim.

## Self-check

Before you submit, confirm each line honestly:

- [✔️] Every section above is filled — markdown thinking AND the code that backs it
- [✔️] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔️] No client names, URLs, or private queries anywhere
- [✔️] My claims use careful words: observed, measured, directional, decision-support
- [✔️] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.